# ARTI 403 – Image Processing
## Lab 5 — Spatial Filtering: Smoothing and Sharpening

**Student name:** Ahmed Albuainain
**Student ID:** 2240006128
**Section:** 7MA1
**Date:** 13 October 2026

---

### Session Outcome
**Outcome #2:** Write a program that implements fundamental image processing algorithms.

### Contents
| Part | Description |
|---|---|
| Part 0 | Environment check and preparing the image |
| Part 1 | What spatial filtering is — the convolution kernel |
| Part 2 | Unsharp masking (the manual's Task 1) |
| Task 1 | Convolve an image with a 7x7 box filter |
| Task 2 | Apply a 5x5 and then a 21x21 Gaussian filter |
| Task 3 | Sharpen with a 3x3 Laplacian filter |

---
## Part 0 — Environment check and preparing the image

The manual reads `images/Parrot.png`. That file is not distributed with the lab, so the cell
below creates it from a colour sample of `skimage.data`. If the file already exists it is kept,
so an image supplied by the instructor takes priority.

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import skimage as ski
from skimage import data

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
print("scikit-image:", ski.__version__)
print("Matplotlib  :", matplotlib.__version__)

%matplotlib inline

In [ ]:
IMG_DIR = os.path.join("..", "images")
if not os.path.isdir(IMG_DIR):
    IMG_DIR = "images"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)

def img_path(name):
    return os.path.join(IMG_DIR, name)

if not os.path.exists(img_path("Parrot.png")):
    cv2.imwrite(img_path("Parrot.png"),
                cv2.cvtColor(data.coffee(), cv2.COLOR_RGB2BGR))

print("images folder:", os.path.abspath(IMG_DIR))
print("contents     :", sorted(os.listdir(IMG_DIR)))

A display helper. Everything in this notebook is drawn with Matplotlib rather than
`cv2.imshow`, which opens a blocking window outside the notebook and can leave the kernel
stuck.

In [ ]:
def show(images, titles, cols=3, size=4.6, suptitle=None, cmap='gray'):
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(axes, images, titles):
        if im.ndim == 3:
            ax.imshow(np.clip(im, 0, 1) if im.dtype != np.uint8 else im)
        else:
            ax.imshow(im, cmap=cmap)
        ax.set_title(t, fontsize=11)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    plt.show()

---
## Part 1 — What spatial filtering actually is

Every operation in this lab is the same mechanism with a different set of numbers.

A small grid of weights — the **kernel** — is slid over the image. At each position, the pixels
under the kernel are multiplied by the corresponding weights and summed, and that single number
becomes the output pixel. This is why these are called **neighbourhood operations**: unlike the
intensity transformations of Lab 4, the output at a pixel depends on its *neighbours*, not only
on itself.

$$g(x,y) = \sum_{s}\sum_{t} w(s,t)\, f(x+s,\, y+t)$$

The weights decide everything:

| Kernel weights | Effect |
|---|---|
| all equal, summing to 1 | **box filter** — plain average, smoothing |
| bell-shaped, summing to 1 | **Gaussian** — weighted average, smoothing |
| summing to 0, centre opposite in sign to neighbours | **Laplacian** — detects change, used for sharpening |

### The two rules that explain the results later

**A kernel that sums to 1 preserves overall brightness.** On a flat region every pixel has the
same value, so the output is that value times the sum of the weights. If the weights summed to
2 the image would double in brightness.

**A kernel that sums to 0 gives zero on flat regions.** This is what makes the Laplacian a
*detector*: it outputs nothing where the image is uniform and reacts only where the intensity
changes — at edges and fine detail.

In [ ]:
box7 = np.ones((7, 7), np.float32) / 49.0
gauss5 = cv2.getGaussianKernel(5, -1) @ cv2.getGaussianKernel(5, -1).T
laplacian_k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)

print("7x7 box filter — every weight identical:")
print(np.round(box7, 4)[:2], "...")
print("sum =", round(float(box7.sum()), 6), "-> preserves brightness")
print()
print("5x5 Gaussian — centre weighted most:")
print(np.round(gauss5, 4))
print("sum =", round(float(gauss5.sum()), 6), "-> preserves brightness")
print()
print("3x3 Laplacian:")
print(laplacian_k)
print("sum =", float(laplacian_k.sum()), "-> zero on flat regions")

### The 1-D profile makes the difference obvious

The centre row of each smoothing kernel, plotted. The box filter gives a neighbour 7 pixels
away exactly as much influence as the pixel itself; the Gaussian tapers off with distance.
That single difference is why the box filter produces harder, blockier results.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(-3, 4)
ax.plot(x, box7[3], 'o-', linewidth=2, label='box 7x7 (centre row)')
g7 = cv2.getGaussianKernel(7, -1).ravel()
ax.plot(x, g7, 's-', linewidth=2, label='Gaussian 7x7 (centre row)')
ax.set_xlabel("distance from centre pixel")
ax.set_ylabel("weight")
ax.set_title("Why a Gaussian looks smoother than a box filter")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2 — Unsharp masking (the manual's Task 1)

Unsharp masking is the classic sharpening technique, and its name is counter-intuitive: it
sharpens an image by **blurring** it first.

The logic is in three steps:

1. **Blur the image.** The blurred version contains only the slowly-varying content — the broad
   shapes, with the fine detail smoothed away.
2. **Subtract it from the original.** What is left, `original - blurred`, is exactly the detail
   the blur removed: edges, texture, fine structure. This difference is the **mask**.
3. **Add the mask back, amplified.** `original + amount x mask` exaggerates that detail, which
   the eye reads as sharpness.

$$g = f + k\,(f - \text{blur}(f))$$

In [ ]:
# Read the image
image = cv2.imread(img_path('Parrot.png'))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)   # Convert from BGR to RGB

# Gaussian sigma
sigma = 2
# parameter
amount = 1.5

# Convert image to float32 for processing
image_float = image.astype(np.float32) / 255.0

# Apply Gaussian blur
blurred = cv2.GaussianBlur(image_float, (0, 0), sigmaX=sigma, sigmaY=sigma)

# Perform Unsharp Masking
sharpened = np.clip(image_float + amount * (image_float - blurred), 0, 1)

# Plot original, blurred, and sharpened images
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(image)
ax[0].set_title("Original Image")
ax[0].axis("off")
ax[1].imshow(blurred)
ax[1].set_title("Blurred Image")
ax[1].axis("off")
ax[2].imshow(sharpened)
ax[2].set_title("Sharpened (Unsharp Masking)")
ax[2].axis("off")
plt.show()

### Why the image is converted to float first

`image.astype(np.float32) / 255.0` is not decoration. The mask `image_float - blurred` contains
**negative values** wherever the original is darker than its blurred version, which is half of
every edge. In `uint8` those negatives wrap around to large positive numbers and the result is
ruined. Working in float keeps them, and `np.clip(..., 0, 1)` folds the range back at the end.

In [ ]:
mask = image_float - blurred

print("mask range  :", round(float(mask.min()), 4), "to", round(float(mask.max()), 4))
print("negative    :", f"{100 * (mask < 0).mean():.1f}% of all values")
print()
boosted = image_float + amount * mask
print("before clipping:", round(float(boosted.min()), 3), "to", round(float(boosted.max()), 3))
print("pixels below 0 :", f"{100 * (boosted < 0).mean():.1f}%")
print("pixels above 1 :", f"{100 * (boosted > 1).mean():.1f}%")

### Seeing the mask itself

The mask is grey wherever the image is flat and bright or dark along edges — a direct picture of
"what the blur threw away". It is shifted by 0.5 below only so the negative values are visible.

In [ ]:
show([image_float, blurred, mask + 0.5, sharpened],
     ["original", f"blurred (sigma = {sigma})",
      "mask = original - blurred\n(shifted by 0.5 to show negatives)",
      f"sharpened (amount = {amount})"],
     cols=4, size=4.2)

### The effect of the two parameters

`sigma` decides *which* detail is treated as detail — a larger sigma blurs more, so the mask
contains coarser structure. `amount` decides how strongly that detail is pushed back in. Too
large an amount produces **halos**: bright and dark outlines along strong edges, the standard
sign of over-sharpening.

In [ ]:
crop = image_float[120:300, 180:420]

variants, titles = [crop], ["original"]
for a in [0.5, 1.5, 3.0, 6.0]:
    b = cv2.GaussianBlur(crop, (0, 0), sigmaX=2, sigmaY=2)
    variants.append(np.clip(crop + a * (crop - b), 0, 1))
    titles.append(f"amount = {a}")

show(variants, titles, cols=5, size=3.4,
     suptitle="Increasing the amount — note the halos appearing at the highest value")

---
## Task 1 — Convolve the image with a 7x7 box filter

A box filter replaces each pixel with the plain average of the 7x7 block around it. The kernel
is built explicitly here and applied with `cv2.filter2D`, which is the general convolution
function — the same call would apply any kernel.

In [ ]:
img_bgr = cv2.imread(img_path('Parrot.png'))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# build the 7x7 box kernel: every weight equal, summing to 1
kernel_7 = np.ones((7, 7), np.float32) / 49.0
print("kernel shape:", kernel_7.shape, "| every weight:", round(float(kernel_7[0, 0]), 6))
print("sum of weights:", round(float(kernel_7.sum()), 6))

# convolve
smoothed = cv2.filter2D(img_rgb, -1, kernel_7)

show([img_rgb, smoothed], ["Original", "Smoothed — 7x7 box filter"], cols=2, size=5.4)

### Checking the result against the built-in function

`cv2.blur` is OpenCV's dedicated box-filter call. If the kernel was built correctly the two
results must be identical, pixel for pixel.

In [ ]:
builtin = cv2.blur(img_rgb, (7, 7))
print("filter2D with the manual kernel == cv2.blur :", np.array_equal(smoothed, builtin))
print("largest difference                          :",
      int(np.abs(smoothed.astype(np.int16) - builtin.astype(np.int16)).max()))
print()
print("mean brightness before :", round(float(img_rgb.mean()), 2))
print("mean brightness after  :", round(float(smoothed.mean()), 2),
      " <- preserved, because the weights sum to 1")
print("std (detail) before    :", round(float(img_rgb.std()), 2))
print("std (detail) after     :", round(float(smoothed.std()), 2), " <- reduced by smoothing")

### The effect of the kernel size

In [ ]:
sizes = [1, 3, 7, 15, 31]
outs, titles = [], []
for k in sizes:
    out = img_rgb if k == 1 else cv2.filter2D(img_rgb, -1, np.ones((k, k), np.float32) / (k * k))
    outs.append(out)
    titles.append(f"{k}x{k}" + ("  (original)" if k == 1 else ""))

show(outs, titles, cols=5, size=3.4,
     suptitle="Box filter — the larger the kernel, the more detail is lost")

for k, out in zip(sizes, outs):
    print(f"{k:>2}x{k:<2}  std = {out.std():6.2f}   mean = {out.mean():6.2f}")

**A note on terminology.** `cv2.filter2D` performs **correlation**, not true convolution: it
does not flip the kernel before applying it. For a symmetric kernel such as a box or a Gaussian
the two are identical, which is why it makes no difference here. For an asymmetric kernel the
results differ, and `cv2.flip(kernel, -1)` would be needed first.

---
## Task 2 — A 5x5 and then a 21x21 Gaussian filter

A Gaussian filter is also an averaging filter, but the weights fall off with distance from the
centre instead of being equal. Nearby pixels count more, which produces a much more natural
blur than the box filter.

In [ ]:
gauss_5 = cv2.GaussianBlur(img_rgb, (5, 5), 0)
gauss_21 = cv2.GaussianBlur(img_rgb, (21, 21), 0)

show([img_rgb, gauss_5, gauss_21],
     ["Original", "Gaussian 5x5", "Gaussian 21x21"],
     cols=3, size=5.0)

for name, im in [("original", img_rgb), ("5x5", gauss_5), ("21x21", gauss_21)]:
    print(f"{name:<10} std = {im.std():6.2f}   mean = {im.mean():6.2f}")

### What the `0` argument actually does

Passing `0` for `sigmaX` tells OpenCV to **derive sigma from the kernel size**, using

$$\sigma = 0.3\,\big((k-1)\cdot 0.5 - 1\big) + 0.8$$

So the two calls above are not just "bigger" and "smaller" — they correspond to genuinely
different amounts of blur. The reverse convention is used in Part 2: passing `(0, 0)` for the
kernel size tells OpenCV to derive the kernel size from the sigma instead.

In [ ]:
for k in [5, 21]:
    s = 0.3 * ((k - 1) * 0.5 - 1) + 0.8
    print(f"ksize = {k:>2}  ->  sigma = {s:.2f}")

print()
explicit = cv2.GaussianBlur(img_rgb, (5, 5), sigmaX=1.1, sigmaY=1.1)
print("ksize=(5,5) sigma=0  vs  ksize=(5,5) sigma=1.1 (the derived value)")
print("largest difference:",
      int(np.abs(gauss_5.astype(np.int16) - explicit.astype(np.int16)).max()), "levels")

The two are close but **not identical**, which is worth understanding rather than assuming.
Printing the kernel OpenCV actually uses for a small window with `sigma = 0` explains it.

In [ ]:
k5 = cv2.getGaussianKernel(5, -1).ravel()     # -1 means "derive from the size"
print("1-D kernel for ksize=5, sigma derived:", k5)
print("multiplied by 16                     :", np.round(k5 * 16, 6))

The weights are exactly 1, 4, 6, 4, 1 divided by 16 — the **binomial** coefficients, not
samples of a Gaussian curve. For small windows OpenCV uses this fixed integer kernel instead of
evaluating the Gaussian formula, because it is a close approximation that is faster and exact in
integer arithmetic. It corresponds to roughly `sigma = 1.1`, which is why the two results differ
by only a few intensity levels rather than being byte-identical.

### Box versus Gaussian at the same kernel size

This is the comparison that matters. Both kernels are 7x7 and both preserve brightness, but the
box filter produces visibly harder, blockier edges because a distant neighbour carries the same
weight as the centre pixel.

In [ ]:
patch = img_rgb[150:280, 220:420]
box_p = cv2.filter2D(patch, -1, np.ones((7, 7), np.float32) / 49)
gauss_p = cv2.GaussianBlur(patch, (7, 7), 0)

show([patch, box_p, gauss_p],
     ["original (detail)", "box 7x7", "Gaussian 7x7"],
     cols=3, size=4.4)

print("std — original:", round(float(patch.std()), 2),
      "| box:", round(float(box_p.std()), 2),
      "| Gaussian:", round(float(gauss_p.std()), 2))

---
## Task 3 — Sharpen with a 3x3 Laplacian filter

The Laplacian is a **second-derivative** operator. The first derivative is large wherever the
intensity is changing; the second derivative is large wherever that *rate of change* itself
changes — which is exactly where edges and fine detail live.

Its kernel sums to zero, so it returns zero on flat regions and reacts only at changes. The
output is therefore not an image to look at on its own: it is a map of where the detail is.

Following the steps given in the manual.

In [ ]:
# 1. Read the image
image3 = cv2.imread(img_path('Parrot.png'))
image3 = cv2.cvtColor(image3, cv2.COLOR_BGR2RGB)

# 2. Convert image to float32
image3_float = image3.astype(np.float32) / 255.0

# 3. Use the cv2.Laplacian() function
laplacian = cv2.Laplacian(image3_float, cv2.CV_32F, ksize=3)

# 4. Sharpen the image
sharpened3 = np.clip(image3_float - laplacian, 0, 1)

# 5. Plot the three images
show([image3_float, laplacian + 0.5, sharpened3],
     ["Original", "Laplacian\n(shifted by 0.5 to show negatives)", "Sharpened"],
     cols=3, size=5.0)

print("Laplacian range:", round(float(laplacian.min()), 3),
      "to", round(float(laplacian.max()), 3))
print("mean           :", round(float(laplacian.mean()), 5), " <- near zero, as expected")

### Why the formula **subtracts** the Laplacian

This is the detail worth getting right, and it comes down to a sign convention.

OpenCV's Laplacian kernel is

$$\begin{bmatrix}0 & 1 & 0\\ 1 & -4 & 1\\ 0 & 1 & 0\end{bmatrix}$$

with a **negative centre**. At a bright point surrounded by darker pixels, the output is
therefore negative. Subtracting a negative number makes the bright point brighter — so
`f - laplacian` enhances peaks and deepens troughs, which is sharpening.

Had the kernel used a positive centre, the correct formula would be `f + laplacian`. Using the
wrong sign raises no error, so the only way to catch it is to check what it does. The cell below
measures the brightest and darkest points of the image directly.

In [ ]:
wrong_sign = np.clip(image3_float + laplacian, 0, 1)

# the strongest 1% of peaks and troughs, as identified by the Laplacian itself
peaks = laplacian < np.percentile(laplacian, 1)      # bright points
troughs = laplacian > np.percentile(laplacian, 99)   # dark points

print(f"{'':<16}{'at bright peaks':>18}{'at dark troughs':>18}")
print("-" * 52)
for name, im in [("original", image3_float),
                 ("f - laplacian", sharpened3),
                 ("f + laplacian", wrong_sign)]:
    print(f"{name:<16}{im[peaks].mean():>18.4f}{im[troughs].mean():>18.4f}")

The numbers settle it. With `f - laplacian` the bright points get brighter and the dark
points get darker — contrast at the detail is increased, which is sharpening. With
`f + laplacian` the two **swap**: bright points go dark and dark points go bright, so edges come
out inverted.

This is why "the standard deviation went up" is not enough to confirm sharpening. Both versions
raise it, because both add energy at the edges. Only the direction of the change tells you which
one is correct.

In [ ]:
print("standard deviation (both rise — so it does not distinguish them)")
for name, im in [("original", image3_float), ("f - laplacian", sharpened3),
                 ("f + laplacian", wrong_sign)]:
    print(f"  {name:<16}{im.std():.4f}")

show([image3_float, sharpened3, wrong_sign],
     ["original", "f - laplacian  (correct)",
      "f + laplacian  (wrong sign — edges inverted)"],
     cols=3, size=4.6)

### The Laplacian amplifies noise

Because it responds to rapid change, the Laplacian cannot tell a genuine edge from a noisy
pixel. On a noisy image it amplifies the noise along with the detail. The standard remedy is to
blur slightly first — the combination is known as the Laplacian of Gaussian.

In [ ]:
rng = np.random.default_rng(0)
noisy = np.clip(image3_float + rng.normal(0, 0.04, image3_float.shape), 0, 1).astype(np.float32)

lap_noisy = cv2.Laplacian(noisy, cv2.CV_32F, ksize=3)
sharp_noisy = np.clip(noisy - lap_noisy, 0, 1)

pre_blur = cv2.GaussianBlur(noisy, (0, 0), sigmaX=1.2, sigmaY=1.2)
lap_log = cv2.Laplacian(pre_blur, cv2.CV_32F, ksize=3)
sharp_log = np.clip(pre_blur - lap_log, 0, 1)

show([noisy, sharp_noisy, sharp_log],
     ["noisy input", "Laplacian applied directly\n(noise amplified)",
      "blur first, then Laplacian\n(Laplacian of Gaussian)"],
     cols=3, size=4.6)

print("noise level (std of the difference from the clean image)")
print("  noisy input          :", round(float((noisy - image3_float).std()), 4))
print("  sharpened directly   :", round(float((sharp_noisy - image3_float).std()), 4))
print("  blurred then sharpened:", round(float((sharp_log - image3_float).std()), 4))

### Unsharp masking versus the Laplacian

Both sharpen, and they are closely related: unsharp masking subtracts a blurred copy, while the
Laplacian subtracts a second-derivative estimate of the same thing. The practical difference is
control — unsharp masking exposes `sigma` and `amount`, so the strength and the scale of the
detail can both be tuned, while the plain Laplacian has a fixed strength.

In [ ]:
um = np.clip(image3_float + 1.5 * (image3_float -
             cv2.GaussianBlur(image3_float, (0, 0), sigmaX=2, sigmaY=2)), 0, 1)

show([image3_float, um, sharpened3],
     ["original", "unsharp masking (sigma=2, amount=1.5)", "Laplacian sharpening"],
     cols=3, size=4.6)

for name, im in [("original", image3_float), ("unsharp mask", um), ("Laplacian", sharpened3)]:
    print(f"{name:<16} std = {im.std():.4f}")

### Summary figure

In [ ]:
summary = [(img_rgb, "original"),
           (cv2.filter2D(img_rgb, -1, np.ones((7, 7), np.float32) / 49), "box 7x7"),
           (gauss_5, "Gaussian 5x5"),
           (gauss_21, "Gaussian 21x21"),
           ((sharpened * 255).astype(np.uint8), "unsharp masking"),
           ((sharpened3 * 255).astype(np.uint8), "Laplacian sharpening")]

show([s[0] for s in summary], [s[1] for s in summary], cols=3, size=4.2,
     suptitle="Lab 5 — smoothing and sharpening")

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (im, t) in zip(axes.flat, summary):
    ax.imshow(im)
    ax.set_title(t, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('output/lab5_summary.png', dpi=110, bbox_inches='tight')
plt.close()
print("Saved: output/lab5_summary.png")

---
## Conclusion

**Spatial filtering is one mechanism with different numbers in it.** A kernel slides over the
image, and each output pixel is a weighted sum of its neighbourhood. Smoothing and sharpening
are not different algorithms — they are different weights. The sum of those weights predicts the
behaviour: a kernel summing to 1 preserves brightness and averages, a kernel summing to 0
returns zero on flat regions and therefore detects change.

**Box versus Gaussian.** Both average and both preserve brightness, but the box filter gives a
distant neighbour exactly as much influence as the centre pixel, which produces harder, blockier
results. The Gaussian tapers with distance and looks natural.

Two details about the Gaussian are easy to assume wrongly. Passing `0` for sigma makes OpenCV
derive it from the kernel size, so a 5x5 and a 21x21 filter correspond to sigma 1.10 and 3.50
respectively. But for a small window OpenCV does not sample the Gaussian curve at all: the 5x5
kernel it uses is exactly the binomial 1, 4, 6, 4, 1 over 16, a fast integer approximation. That
is why passing `sigma = 1.1` explicitly gives a result close to, but not identical with, the
derived one.

**Sharpening works by subtraction.** Unsharp masking blurs the image, subtracts the blur to
isolate the detail the blur removed, and adds that detail back amplified. Pushing `amount` too
far produces halos along strong edges. The Laplacian reaches the same goal through a second
derivative, and the sign of the formula follows from the kernel: OpenCV's centre weight is
negative, so `f - laplacian` sharpens. Using `f + laplacian` raises no error — measured at the
strongest peaks and troughs, it **inverts** them, sending bright detail dark and dark detail
bright. Notably the standard deviation rises in both cases, so that statistic alone cannot tell
the correct version from the wrong one; only the direction of the change can.

**Two practical points recurred.** Everything was computed in `float32` because the sharpening
mask is negative across half of every edge and would wrap around in `uint8`, with `np.clip` used
once at the end. And the Laplacian cannot distinguish an edge from noise, so on a noisy image it
amplifies both — blurring slightly first (the Laplacian of Gaussian) is the standard remedy.

---

### Resources
- https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html
- https://datacarpentry.github.io/image-processing/06-blurring.html